# Trabajo práctico: búsqueda voraz y A* sobre un grafo dirigido

**Estudiante:** Agustina Waigel  
**Materia:** Inteligencia Artificial

Se implementan costo uniforme (UCS), búsqueda voraz y A* con un mismo esquema. El nodo conserva los campos del código inicial de la consigna; la frontera usa `heapq` y respeta los empates por orden de inserción.

## 1. Grafo, heurística y código inicial

In [1]:
from heapq import heappush, heappop
from itertools import count

grafo = {
    "S": [("A", 2), ("B", 2)],
    "A": [("C", 2), ("D", 5)],
    "B": [("D", 2)],
    "C": [("G", 3)],
    "D": [("G", 6)],
    "G": [],
}

heuristica = {"S": 7, "A": 5, "B": 7, "C": 3, "D": 6, "G": 0}

# Se conserva la representación de nodos mediante diccionarios dada en la consigna.
def crear_nodo(estado, padre=None, accion=None, g=0, h=0):
    return {
        "estado": estado,
        "padre": padre,
        "accion": accion,
        "g": g,
        "h": h,
        "f": g + h,
    }

raiz = crear_nodo("S", g=0, h=heuristica["S"])
print("Nodo inicial:", {k: v for k, v in raiz.items() if k != "padre"})

Nodo inicial: {'estado': 'S', 'accion': None, 'g': 0, 'h': 7, 'f': 7}


## 2. Búsqueda común

`generados` cuenta los nodos insertados, incluida la raíz. `expandidos` cuenta los estados a los que se les examinan sucesores: no incluye al objetivo. `reaperturas` sigue la definición de la consigna: mejoras del costo de un estado previamente conocido. La frontera mostrada en cada traza es la frontera válida **después** de expandir el estado de esa fila, ordenada por prioridad y luego por inserción. Las entradas obsoletas se descartan.

In [2]:
def reconstruir_camino(nodo):
    camino = []
    while nodo is not None:
        camino.append(nodo["estado"])
        nodo = nodo["padre"]
    return list(reversed(camino))


def prioridad(nodo, algoritmo):
    if algoritmo == "UCS":
        return nodo["g"]
    if algoritmo == "Voraz":
        return nodo["h"]
    if algoritmo == "A*":
        return nodo["f"]
    raise ValueError(f"Algoritmo desconocido: {algoritmo}")


def buscar(algoritmo, inicio="S", objetivo="G"):
    orden = count()
    raiz = crear_nodo(inicio, g=0, h=heuristica[inicio])
    frontera = []  # (prioridad, orden_de_insercion, nodo)
    heappush(frontera, (prioridad(raiz, algoritmo), next(orden), raiz))
    mejor_g = {inicio: 0}
    generados = 1
    expandidos = 0
    frontera_maxima = 1
    reaperturas = 0
    traza = []

    def frontera_vigente():
        # Entradas viejas pueden seguir físicamente en el heap, pero no son frontera válida.
        return sorted(
            (p, i, n) for p, i, n in frontera
            if n["g"] == mejor_g[n["estado"]]
        )

    while frontera:
        p, _, nodo = heappop(frontera)
        estado = nodo["estado"]
        if nodo["g"] != mejor_g[estado]:
            continue

        # El objetivo se comprueba al EXTRAER, nunca al generar.
        if estado == objetivo:
            traza.append((estado, p, [(n["estado"], pr) for pr, _, n in frontera_vigente()]))
            return {
                "algoritmo": algoritmo,
                "camino": reconstruir_camino(nodo),
                "costo": nodo["g"],
                "generados": generados,
                "expandidos": expandidos,
                "frontera_maxima": frontera_maxima,
                "reaperturas": reaperturas,
                "traza": traza,
            }

        expandidos += 1
        for sucesor, costo_arista in grafo[estado]:
            nuevo_g = nodo["g"] + costo_arista
            if nuevo_g < mejor_g.get(sucesor, float("inf")):
                if sucesor in mejor_g:
                    reaperturas += 1
                mejor_g[sucesor] = nuevo_g
                nuevo = crear_nodo(
                    sucesor, padre=nodo, accion=f"{estado} -> {sucesor}",
                    g=nuevo_g, h=heuristica[sucesor]
                )
                heappush(frontera, (prioridad(nuevo, algoritmo), next(orden), nuevo))
                generados += 1

        vigente = frontera_vigente()
        frontera_maxima = max(frontera_maxima, len(vigente))
        traza.append((estado, p, [(n["estado"], pr) for pr, _, n in vigente]))

    return {
        "algoritmo": algoritmo, "camino": None, "costo": None,
        "generados": generados, "expandidos": expandidos,
        "frontera_maxima": frontera_maxima, "reaperturas": reaperturas,
        "traza": traza,
    }

## 3. Ejecutar los tres algoritmos y mostrar las trazas

In [3]:
resultados = {nombre: buscar(nombre) for nombre in ("UCS", "Voraz", "A*")}

for nombre, resultado in resultados.items():
    print(f"\n=== {nombre} ===")
    print("Paso | Extraído (prioridad) | Frontera después de expandir")
    for paso, (estado, p, frontera) in enumerate(resultado["traza"], 1):
        vista = ", ".join(f"{e}({pr})" for e, pr in frontera) or "vacía"
        print(f"{paso:>4} | {estado} ({p}) | {vista}")
    print("Camino:", " → ".join(resultado["camino"]) if resultado["camino"] else "sin solución")
    print("Costo:", resultado["costo"])
    print("Generados:", resultado["generados"],
          "| Expandidos:", resultado["expandidos"],
          "| Frontera máxima:", resultado["frontera_maxima"],
          "| Reaperturas:", resultado["reaperturas"])


=== UCS ===
Paso | Extraído (prioridad) | Frontera después de expandir
   1 | S (0) | A(2), B(2)
   2 | A (2) | B(2), C(4), D(7)
   3 | B (2) | C(4), D(4)
   4 | C (4) | D(4), G(7)
   5 | D (4) | G(7)
   6 | G (7) | vacía
Camino: S → A → C → G
Costo: 7
Generados: 7 | Expandidos: 5 | Frontera máxima: 3 | Reaperturas: 1

=== Voraz ===
Paso | Extraído (prioridad) | Frontera después de expandir
   1 | S (7) | A(5), B(7)
   2 | A (5) | C(3), D(6), B(7)
   3 | C (3) | G(0), D(6), B(7)
   4 | G (0) | D(6), B(7)
Camino: S → A → C → G
Costo: 7
Generados: 6 | Expandidos: 3 | Frontera máxima: 3 | Reaperturas: 0

=== A* ===
Paso | Extraído (prioridad) | Frontera después de expandir
   1 | S (7) | A(7), B(9)
   2 | A (7) | C(7), B(9), D(13)
   3 | C (7) | G(7), B(9), D(13)
   4 | G (7) | B(9), D(13)
Camino: S → A → C → G
Costo: 7
Generados: 6 | Expandidos: 3 | Frontera máxima: 3 | Reaperturas: 0


## 4. Tabla comparativa

In [4]:
prioridades = {"UCS": "g", "Voraz": "h", "A*": "g+h"}
filas = [
    ("Camino", lambda r, n: " → ".join(r["camino"])),
    ("Costo", lambda r, n: r["costo"]),
    ("Prioridad", lambda r, n: prioridades[n]),
    ("Expandidos antes de extraer G", lambda r, n: r["expandidos"]),
    ("Generados", lambda r, n: r["generados"]),
    ("Frontera máxima", lambda r, n: r["frontera_maxima"]),
    ("Reaperturas", lambda r, n: r["reaperturas"]),
]
print("| Resultado | UCS | Voraz | A* |")
print("|---|---|---|---|")
for etiqueta, extraer in filas:
    print("| " + etiqueta + " | " + " | ".join(
        str(extraer(resultados[nombre], nombre)) for nombre in ("UCS", "Voraz", "A*")
    ) + " |")

| Resultado | UCS | Voraz | A* |
|---|---|---|---|
| Camino | S → A → C → G | S → A → C → G | S → A → C → G |
| Costo | 7 | 7 | 7 |
| Prioridad | g | h | g+h |
| Expandidos antes de extraer G | 5 | 3 | 3 |
| Generados | 7 | 6 | 6 |
| Frontera máxima | 3 | 3 | 3 |
| Reaperturas | 1 | 0 | 0 |


## 5. Preguntas de análisis

1. **¿Por qué coinciden voraz y A*?** Desde `S`, voraz prefiere `A` porque `h(A)=5 < h(B)=7`; A* también prefiere `A` porque ambos tienen `g=2`, y entonces `f(A)=7 < f(B)=9`. Desde `A`, ambos eligen `C` antes que `D` (`h(C)=3 < h(D)=6`; `f(C)=7 < f(D)=13`). Así llegan a `G` por `S → A → C → G`, de costo 7. La heurística guía ambas decisiones por esa rama, que en este grafo resulta ser la más barata. Además, `h` no sobreestima el costo restante real de ningún estado: es admisible (aunque no consistente en `S → A`). Que coincidan aquí no da a voraz una garantía general.

2. **¿Voraz garantiza el menor costo en general?** No. Su prioridad es únicamente `h`: ignora el costo `g` ya recorrido. Por eso puede extraer un objetivo por una ruta cara antes de explorar otra ruta más barata.

3. **¿Qué pasa con `h=0` en A*?** Su prioridad queda `f=g+0=g`; se comporta como UCS, incluidos los empates si se conserva el mismo orden de inserción.

4. **¿Hubo reaperturas?** En UCS, `D` se descubre desde `A` con `g=7` y después mejora desde `B` a `g=4`: hay una mejora. En voraz, primero se descubre `G` desde `C` con `g=7` y se termina antes de mejorar otros estados: no hay ninguna. En A* se llega a `G` por `C` antes de extraer `B`, así que tampoco hay mejoras. Usamos “reapertura” con la definición de la consigna (mejorar `mejor_g` de un estado ya conocido); en UCS `D` aún no había sido expandido cuando mejoró.

5. **¿Expandir menos implica un camino más barato?** No. En este caso voraz expande menos que UCS y ambos entregan costo 7, pero el número de expansiones mide trabajo de búsqueda, no calidad de la ruta. UCS considera los costos acumulados y, con aristas de costo no negativo, garantiza el costo mínimo; voraz no tiene esa garantía.